# Open Loop Lab — Differential Drive Robot

This notebook simulates the trajectory of a differential drive robot under open-loop kinematic control, comparing an ideal model against one affected by mechanical backlash in the wheel-shaft interface.

The central question: what happens to trajectory geometry when backlash is modeled as a deadband nonlinearity applied uniformly to both wheels?

**Sections**
1. Dependencies
2. Robot Parameters
3. Backlash Model
4. Simulation
5. Results
6. Interpretation

## 1. Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Robot Parameters

Physical parameters measured from the hardware build.

> **Note:** The accompanying report uses `r = 0.05 m` and `b = 0.20 m` as illustrative values for worked examples. The values below reflect the actual geometry of the physical robot.

| Parameter | Symbol | Value |
|---|---|---|
| Wheel radius | r | 0.03355 m |
| Track width (axle to axle) | b | 0.163 m |
| Backlash deadband | θ_b | 15° = 0.2618 rad |
| Simulation timestep | Δt | 0.01 s |
| Simulation duration | T | 10 s |

In [ ]:
# --- Robot geometry ---
r       = 0.03355               # Wheel radius (m)
b       = 0.163                 # Track width (m)
theta_b = np.deg2rad(15)        # Backlash deadband (rad)

# --- Simulation time ---
dt = 0.01
T  = 10
t  = np.arange(0, T, dt)

# --- Wheel inputs (rad/s) ---
# Differential inputs produce a curved trajectory under ideal conditions.
wr = 10 * np.ones_like(t)      # Right wheel
wl = 8  * np.ones_like(t)      # Left wheel

### Derived Velocities

Before running the simulation, we compute the forward and angular velocities analytically for both models. This makes the core insight visible without simulation:

- Backlash applies the same offset `θ_b` to both wheels.
- The velocity **difference** `(ωr − ωl)` is unchanged → angular velocity `ω` is preserved.
- The velocity **sum** `(ωr + ωl)` decreases → forward velocity `v` is reduced.
- Since `R = v / ω`, a reduced `v` with constant `ω` contracts the turning radius.

In [ ]:
# --- Analytical comparison before simulation ---
v_ideal    = (r / 2) * (wr[0] + wl[0])
w_ideal    = (r / b) * (wr[0] - wl[0])
R_ideal    = v_ideal / w_ideal

v_backlash = (r / 2) * ((wr[0] - theta_b) + (wl[0] - theta_b))
w_backlash = (r / b) * ((wr[0] - theta_b) - (wl[0] - theta_b))  # Difference unchanged
R_backlash = v_backlash / w_backlash

print(f"{'Model':<12} | {'v (m/s)':<12} | {'ω (rad/s)':<12} | {'R (m)':<10}")
print("-" * 54)
print(f"{'Ideal':<12} | {v_ideal:<12.4f} | {w_ideal:<12.4f} | {R_ideal:<10.4f}")
print(f"{'Backlash':<12} | {v_backlash:<12.4f} | {w_backlash:<12.4f} | {R_backlash:<10.4f}")
print()
print(f"Forward velocity reduction due to backlash : {v_ideal - v_backlash:.4f} m/s")
print(f"Angular velocity change due to backlash    : {abs(w_ideal - w_backlash):.6f} rad/s (preserved)")
print(f"Turning radius contraction                 : {R_ideal - R_backlash:.4f} m")

## 3. Backlash Model

Backlash in the wheel-shaft interface is modeled as a **deadband nonlinearity**. Below the backlash threshold `θ_b`, the commanded velocity produces no effective motion. Above it, the effective velocity is the commanded velocity offset by the deadband magnitude:

```
ω_eff = 0                          if |ω| ≤ θ_b
ω_eff = ω − sgn(ω) × θ_b          if |ω| > θ_b
```

Since both wheel commands satisfy `|ω| >> θ_b`, the deadband acts as a constant offset throughout the simulation.

In [ ]:
def apply_backlash(w, theta_b):
    """
    Models wheel-shaft backlash as a deadband nonlinearity.
    Returns zero velocity within the deadband, offset velocity outside it.

    Parameters
    ----------
    w : np.ndarray
        Commanded angular velocity array (rad/s).
    theta_b : float
        Backlash threshold (rad). Corresponds to the angular slack
        in the wheel-shaft mechanical interface.

    Returns
    -------
    np.ndarray
        Effective angular velocity after backlash is applied.
    """
    return np.where(np.abs(w) > theta_b, w - np.sign(w) * theta_b, 0.0)


wr_eff = apply_backlash(wr, theta_b)
wl_eff = apply_backlash(wl, theta_b)

print(f"Right wheel: commanded = {wr[0]:.4f} rad/s  |  effective = {wr_eff[0]:.4f} rad/s")
print(f"Left  wheel: commanded = {wl[0]:.4f} rad/s  |  effective = {wl_eff[0]:.4f} rad/s")

## 4. Simulation

Both trajectories are propagated in discrete time using **forward Euler integration**:

```
x[k+1]   = x[k]   + v[k] * cos(φ[k]) * Δt
y[k+1]   = y[k]   + v[k] * sin(φ[k]) * Δt
φ[k+1]   = φ[k]   + ω[k] * Δt
```

Positional error between the two trajectories is measured at each timestep as Euclidean distance:

```
e(t_k) = sqrt( (x_i - x_b)^2 + (y_i - y_b)^2 )
```

In [ ]:
# --- Initialise pose for both models ---
x_i, y_i, phi_i = 0.0, 0.0, 0.0   # Ideal
x_b, y_b, phi_b = 0.0, 0.0, 0.0   # Backlash-affected

traj_i = []
traj_b = []
error  = []

# --- Simulation loop ---
for i in range(len(t)):

    # Ideal model
    v_i   = (r / 2) * (wr[i] + wl[i])
    w_i   = (r / b) * (wr[i] - wl[i])

    x_i   += v_i * np.cos(phi_i) * dt
    y_i   += v_i * np.sin(phi_i) * dt
    phi_i += w_i * dt

    traj_i.append([x_i, y_i])

    # Backlash-affected model
    v_b   = (r / 2) * (wr_eff[i] + wl_eff[i])
    w_b   = (r / b) * (wr_eff[i] - wl_eff[i])

    x_b   += v_b * np.cos(phi_b) * dt
    y_b   += v_b * np.sin(phi_b) * dt
    phi_b += w_b * dt

    traj_b.append([x_b, y_b])

    # Positional error
    error.append(np.sqrt((x_i - x_b)**2 + (y_i - y_b)**2))

traj_i = np.array(traj_i)
traj_b = np.array(traj_b)
error  = np.array(error)

print(f"Simulation complete. Steps: {len(t)}  |  Duration: {T}s  |  Timestep: {dt}s")
print(f"Peak positional error : {error.max():.4f} m  at t = {t[error.argmax()]:.2f}s")
print(f"Final positional error: {error[-1]:.4f} m  at t = {t[-1]:.2f}s")

## 5. Results

### Trajectory Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.plot(traj_i[:, 0], traj_i[:, 1], label="Ideal", linewidth=2)
ax.plot(traj_b[:, 0], traj_b[:, 1], linestyle="--", label="With Backlash", linewidth=2)

# Mark start point
ax.plot(0, 0, 'ko', markersize=6, label="Start (0, 0)")

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Trajectory Comparison")
ax.legend()
ax.grid(True)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### Error Growth Over Time

In [ ]:
peak_idx = error.argmax()

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(t, error, linewidth=2)

# Annotate peak error
ax.annotate(
    f"Peak: {error[peak_idx]:.4f} m",
    xy=(t[peak_idx], error[peak_idx]),
    xytext=(t[peak_idx] - 2.5, error[peak_idx] - 0.006),
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=9
)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Position Error (m)")
ax.set_title("Error Growth Due to Backlash")
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. Interpretation

The trajectory comparison confirms the analytical prediction. Backlash applies a uniform offset `θ_b` to both wheel velocities, which produces an asymmetric effect on the kinematic model:

- The velocity **difference** `(ωr − ωl)` is unchanged, so angular velocity `ω` is preserved.
- The velocity **sum** `(ωr + ωl)` decreases, so forward velocity `v` is reduced.
- Since `R = v / ω`, the reduced `v` contracts the turning radius, producing the inward deviation visible in the trajectory plot.

The error plot shows two phases:

1. **Growth phase**: Error increases rapidly as positional offset accumulates due to the reduced forward velocity.
2. **Convergence phase**: The rate of error growth slows as the two trajectories partially re-converge geometrically, driven by their shared angular velocity.

The residual error at `t = 10s` remains non-negligible. In an open-loop system with no feedback, this error is unbounded over extended operation — it does not self-correct.

**Key takeaway:** Backlash is a deterministic, structured disturbance with predictable geometric consequences. It is not random noise. Its effect can be derived analytically from the kinematic model, which makes it a useful case study for understanding the failure modes of open-loop control before introducing encoder-based correction.